# Étape 4 - Feature engineering sur le jeu de prédiction (online)

Le jeu de prédiction est **dans le futur** : il n'existe pas encore, et surtout
il n'a pas d'historique pour calculer `lag_1W` (le CA d'il y a une semaine).

## Deux façons de calculer le lag

| Méthode | Principe | Inconvénient |
|---|---|---|
| coûteuse en RAM | concaténer **tout** `train` + `future` puis `shift` | occupe beaucoup de mémoire pour peu d'info utile |
| **recommandée** (`lag_online`) | charger seulement la **semaine passée** (`past`), concaténer avec `future`, puis `shift` | aucun — on n'a besoin que d'une semaine |

## Les 4 briques

| Fonction | Rôle |
|---|---|
| `span_future` | génère les dates futures à prédire (1 semaine, au pas horaire) |
| `dummy_day` | jour de la semaine en 6 binaires |
| `hour_cos_sin` | heure en cos/sin |
| `lag_online` | `lag_1W` calculé à partir de `past` uniquement |

`span_future` est autonome ; les 3 autres sont enchaînées par **`features_online`**.

## 1. Importer les librairies

In [1]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:156: FutureWarning: Model's `predict` method contains invalid parameters: {'X'}. Only the following parameter names are allowed: context, model_input, and params. Note that invalid parameters will no longer be permitted in future versions.
  param_names = _check_func_signature(func, "predict")


************************************************************
USING default value : foodcast.settings.dev
************************************************************


## 2. Construire `past` : la semaine juste avant la prédiction

On prédit la semaine qui suit la semaine 200 → `past` = semaine 200, nettoyée avec `etl`.

In [2]:
past = etl(settings.DATA_DIR, 200, 200)
past.tail()

2026-09-09 15:29:45 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (406, 6)
2026-09-09 15:29:45 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (631, 6)
2026-09-09 15:29:45 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (72, 3)
2026-09-09 15:29:45 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (103, 3)
2026-09-09 15:29:45 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - merge: shape = (175, 2)
2026-09-09 15:29:45 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - resample: shape = (148, 2)
2026-09-09 15:29:45 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO 

,order_date,cash_in
143,2018-11-04 15:00:00,0.00
144,2018-11-04 16:00:00,16.85
145,2018-11-04 17:00:00,0.00
146,2018-11-04 18:00:00,38.70
147,2018-11-04 19:00:00,41.40


## 3. Regarder le code de `span_future`

In [3]:
span_future??

Signature: span_future(start: pandas.Timestamp, delta: str = '1W', freq: str = '1h') -> pandas.DataFrame
Source:   
@log_return_shape
def span_future(start: pd.Timestamp, delta: str = '1W', freq: str = '1h') -> pd.DataFrame:
    """
    Generate a dataframe of dates to predict on.
    The future always begins at midnight (after start).
    The future always ends before midnight (after start + delta).

    Parameters
    ----------
    start : pd.Timestamp
        Starting timestamp to predict after.
    delta : str
        Time offset to add from start, by default '1W' (one week).
    freq : str
        New dates frequency sampling, by default '1H4 (one hour).

    Returns
    -------
    pd.DataFrame
        One-column dataframe with date to predict on.
    """
    start = start + pd.Timedelta('1D')
    start = start.normalize()
    end = start + pd.Timedelta(delta) - pd.Timedelta(freq)
    future = pd.date_range(start, end, freq=freq)
    future = future.to_frame(name='order_date')
 

## 4. Générer les dates futures

On part de la dernière date connue dans `past`. `span_future` démarre au minuit suivant
et couvre 1 semaine au pas d'1 heure.

In [4]:
future = span_future(past['order_date'].max())
future.head()

2026-09-09 15:29:50 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/forecast.py - INFO - span_future: shape = (168, 1)


,order_date
0,2018-11-05 00:00:00
1,2018-11-05 01:00:00
2,2018-11-05 02:00:00
3,2018-11-05 03:00:00
4,2018-11-05 04:00:00


In [5]:
future.shape  # 7 jours x 24 heures = 168 lignes

(168, 1)

## 5. Regarder le code de `features_online`

In [6]:
features_online??

Signature:
features_online(
    df: pandas.DataFrame,
    past: pandas.DataFrame,
    degree: int = 1,
    lag_in_week: int = 1,
) -> pandas.DataFrame
Source:   
@log_return_shape
def features_online(df: pd.DataFrame, past: pd.DataFrame, degree: int = 1, lag_in_week: int = 1) -> pd.DataFrame:
    """
    Online feature engineering on a data slice without enough history to compute lags.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe to add features on.
    past : pd.DataFrame
        Data directly in the past of df.
    degree : int, optional
        Degree of the sines and cosines computed, by default 1.
    lag_in_week : int, optional
        Number of weeks to lag, by default 1.

    Returns
    -------
    pd.DataFrame
        Input dataframe with additional features.
    """
    df = dummy_day(df)
    df = hour_cos_sin(df, degree=degree)
    df = lag_online(df, past, lag_in_week=lag_in_week)
    return df
File:      ~/Documents/ml_data_base/MlOps_1/foo

## 6. Construire le jeu de prédiction complet

`features_online(df, past)` : `df` = les dates futures, `past` = la semaine précédente
pour le lag.

In [7]:
future = features_online(future, past)
future.head(20)

2026-09-09 15:29:57 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/feature_engineering.py - INFO - dummy_day: shape = (168, 7)
2026-09-09 15:29:57 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/feature_engineering.py - INFO - hour_cos_sin: shape = (168, 9)
2026-09-09 15:29:57 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/feature_engineering.py - INFO - lag_online: shape = (168, 10)
2026-09-09 15:29:57 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/feature_engineering.py - INFO - features_online: shape = (168, 10)


,order_date,day_1,day_2,day_3,day_4,day_5,day_6,hour_cos_1,hour_sin_1,lag_1W
0,2018-11-05 00:00:00,False,False,False,False,False,False,1.000000e+00,0.000000e+00,0.00
1,2018-11-05 01:00:00,False,False,False,False,False,False,9.659258e-01,2.588190e-01,0.00
2,2018-11-05 02:00:00,False,False,False,False,False,False,8.660254e-01,5.000000e-01,0.00
3,2018-11-05 03:00:00,False,False,False,False,False,False,7.071068e-01,7.071068e-01,0.00
4,2018-11-05 04:00:00,False,False,False,False,False,False,5.000000e-01,8.660254e-01,0.00
5,2018-11-05 05:00:00,False,False,False,False,False,False,2.588190e-01,9.659258e-01,0.00
6,2018-11-05 06:00:00,False,False,False,False,False,False,6.123234e-17,1.000000e+00,0.00
7,2018-11-05 07:00:00,False,False,False,False,False,False,-2.588190e-01,9.659258e-01,0.00
8,2018-11-05 08:00:00,False,False,False,False,False,False,-5.000000e-01,8.660254e-01,0.00
9,2018-11-05 09:00:00,False,False,False,False,False,False,-7.071068e-01,7.071068e-01,0.00


## 7. Vérifier une ligne à la main

On regarde le 5 novembre 2018 à 18h : `lag_1W` doit valoir le `cash_in`
du 29 octobre 2018 à 18h (présent dans `past`).

In [8]:
future[future['order_date'] == '2018-11-05 18:00:00']

,order_date,day_1,day_2,day_3,day_4,day_5,day_6,hour_cos_1,hour_sin_1,lag_1W
18,2018-11-05 18:00:00,False,False,False,False,False,False,-1.836970e-16,-1.0,230.75


In [9]:
resample_past = past.set_index('order_date')
resample_past.loc['2018-10-29 18:00:00', 'cash_in']  # doit correspondre au lag_1W ci-dessus

np.float64(230.75)

## 8. Mettre la date dans l'index

Comme pour `x_train`, on garde la date en index (et non en colonne).

In [10]:
future = future.set_index('order_date')
future.head()

,day_1,day_2,day_3,day_4,day_5,day_6,hour_cos_1,hour_sin_1,lag_1W
order_date,,,,,,,,,
2018-11-05 00:00:00,False,False,False,False,False,False,1.000000,0.000000,0.0
2018-11-05 01:00:00,False,False,False,False,False,False,0.965926,0.258819,0.0
2018-11-05 02:00:00,False,False,False,False,False,False,0.866025,0.500000,0.0
2018-11-05 03:00:00,False,False,False,False,False,False,0.707107,0.707107,0.0
2018-11-05 04:00:00,False,False,False,False,False,False,0.500000,0.866025,0.0


`future` a maintenant exactement les mêmes colonnes que `x_train`.

➡️ Étape suivante : `05_prevision_visualisation.ipynb`